In [ ]:
#@title Setup
# DATA_DIR: folder with the input tables from Dryad (https://doi.org/10.5061/dryad.1vhhmgr88). The fine-tuning panels
# also need the ProteinGym substitution benchmark there (DMS_substitutions.csv and the DMS/ folder of assays).
# Figures and intermediate tables are written to the same folder. On Colab, mount Google Drive first if the data are there.
# Figures use Arial: put arial.ttf in DATA_DIR (Arial cannot be redistributed), otherwise the default sans-serif font is used.
!pip install -q pypalettes

import os
import matplotlib.font_manager as fm

DATA_DIR = "HuSC_data"
os.chdir(DATA_DIR)
ARIAL = "arial.ttf" if os.path.exists("arial.ttf") else fm.findfont("DejaVu Sans")

# Plot settings

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import pypalettes as pp

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

def apply_global_font(ax, size=26):
    """Apply Arial font and consistent size to axes labels and ticks."""
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontproperties(arial_font)
        label.set_fontsize(size)
    for axis_label in [ax.xaxis.label, ax.yaxis.label]:
        axis_label.set_fontproperties(arial_font)
        axis_label.set_fontsize(size)

In [ ]:
# --- Global Plot Settings ---
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.font_manager as fm

# Load font
arial_font = fm.FontProperties(fname=ARIAL)

# --- Global rcParams ---
plt.rcParams.update({
    # Font and text
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial'],
    'font.size': 20,
    'axes.titlesize': 24,
    'axes.labelsize': 26,
    'axes.labelweight': 'bold',
    'xtick.labelsize': 22,
    'ytick.labelsize': 22,
    'axes.linewidth': 2.5,

    # Remove gridlines globally
    'axes.grid': False,
    'grid.linestyle': '',

    # Axes and ticks (ensures they are visible)
    'axes.edgecolor': 'black',
    'xtick.bottom': True,
    'ytick.left': True,
    'xtick.top': False,
    'ytick.right': False,
    'xtick.color': 'black',
    'ytick.color': 'black',
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'xtick.major.width': 2.5,
    'ytick.major.width': 2.5,
    'xtick.major.size': 20,
    'ytick.major.size': 20,

    # Figure defaults
    'figure.figsize': (8, 8),
    'savefig.format': 'svg',
    'savefig.bbox': 'tight',
    'savefig.dpi': 300,

    # Lines
    'lines.linewidth': 2,

    # Legend
    'legend.fontsize': 20,
    'legend.frameon': False
})

# Clean Seaborn base theme (no gridlines)
sns.set_theme(style="white", rc=plt.rcParams)

# Figures

In [ ]:
#@title S1
# Protein outcomes of the full HuSC run (AlphaFold DB v6; 20,504 input UniProt proteins): proteins scored and the
# reason each other protein was skipped, counted from the run logs (every input protein is accounted for).
# Skip reasons in pipeline order (husc/main.py); "No synonymous variant data" = empty_af_dist.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pypalettes import load_cmap

# Configuration
FIGURE_SIZE = (6, 6)
DPI = 300
FONT_SIZE_LEGEND = 18

arial_font = fm.FontProperties(fname=ARIAL, size=FONT_SIZE_LEGEND)

df = pd.DataFrame({
    "category": ["Successfully mapped",
                 "No gnomAD variant data",
                 "No Ensembl transcript IDs mapped",
                 "No valid coding sequence found",
                 "Missing mutation probability data",
                 "Missing AlphaFold structure",
                 "No synonymous variant data"],
    "count": [17020, 1250, 1013, 906, 164, 150, 1],
})
assert pd.read_csv("husc_protein_stats_2026-10.csv")["uniprot_id"].nunique() == df["count"].iloc[0]

total = df["count"].sum()
df["percentage"] = df["count"] / total * 100

# Royal1 + the AsteroidCity2 colors that are distinct from it (tan, dark teal, mint)
asteroid = load_cmap("AsteroidCity2").colors
colors = list(load_cmap("Royal1").colors) + [asteroid[1], asteroid[4], asteroid[2]]

fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)
wedges, _ = ax.pie(
    df["count"],
    startangle=90,
    counterclock=False,
    colors=colors,
    wedgeprops={'linewidth': 2, 'edgecolor': 'white'}
)

legend_labels = [f'{r.category}: {r.count:,} ({r.percentage:.1f}%)' if r.percentage >= 0.1
                 else f'{r.category}: {r.count:,} (<0.1%)'
                 for r in df.itertuples()]
ax.legend(wedges, legend_labels, loc="center left", bbox_to_anchor=(1, 0, 0.5, 1),
          prop=arial_font, frameon=False)
ax.axis('equal')

plt.savefig("S1_protein_mapping_2026-10.svg", dpi=DPI, bbox_inches='tight')
plt.show()

print(f"Total input proteins: {total:,}")
for r in df.itertuples():
    print(f"  {r.category}: {r.count:,} ({r.percentage:.2f}%)")

In [ ]:
#@title S2A: missense HuSC and synonymous score distributions at 5, 8, 11 and 14 Å
# S2_score_histograms_2026-10.csv: histogram counts (bin width 0.1) of missense HuSC and the synonymous score (Phi^-1 of the
# synonymous permutation mid-p, same null and neighborhood) for every residue of the HuSC runs at each radius
# (17,020 proteins; 9,230,134 residues at 8/11/14 Å, 9,217,957 at 5 Å, where start-Met residues with no neighbors drop out).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pypalettes import load_cmap

FIGURE_SIZE = (20, 5.5)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 16
FONT_SIZE_LEGEND = 15
FONT_SIZE_TEXT = 18
SPINE_WIDTH = 1.2
RADII = [5, 8, 11, 14]
HSC_CUTOFF = -2.326

arial_font = fm.FontProperties(fname=ARIAL)
colors = load_cmap('Royal1').colors
KINDS = {'Missense': ('Missense (HuSC)', colors[1]), 'Synonymous': ('Synonymous', colors[0])}

hist = pd.read_csv("S2_score_histograms_2026-10.csv")
fig, axes = plt.subplots(1, 4, figsize=FIGURE_SIZE, dpi=DPI, sharey=True)
for ax, r in zip(axes, RADII):
    for kind, (label, color) in KINDS.items():
        h = hist[(hist.radius == r) & (hist.kind == kind)]
        dens = h['count'] / (h['count'].sum() * (h.bin_right - h.bin_left))
        ax.stairs(dens.values, np.r_[h.bin_left.values, h.bin_right.values[-1]], fill=True, color=color, alpha=0.35)
        ax.stairs(dens.values, np.r_[h.bin_left.values, h.bin_right.values[-1]], color=color, linewidth=2, label=label)
        print(f"{r:2d} Å {kind:10s} n = {h['count'].sum():,}; at the permutation floor (bin < -3.8): "
              f"{h.loc[h.bin_right <= -3.8, 'count'].sum() / h['count'].sum():.2%}")
    ax.axvline(HSC_CUTOFF, color='black', linestyle='--', linewidth=SPINE_WIDTH, zorder=0)
    ax.text(0.04, 0.96, f"{r} Å", transform=ax.transAxes, ha='left', va='top', fontproperties=arial_font, fontsize=FONT_SIZE_TEXT,
            fontweight='bold')
    ax.set_xlim(-4, 4)
    ax.set_xticks([-4, -2, 0, 2, 4])
    ax.set_xlabel("Score", fontsize=FONT_SIZE_LABELS, fontweight='bold', fontproperties=arial_font)
    ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE_TICKS, width=SPINE_WIDTH, bottom=True, left=True)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontproperties(arial_font)
        label.set_fontsize(FONT_SIZE_TICKS)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_linewidth(SPINE_WIDTH)
        ax.spines[spine].set_color('black')
axes[0].set_ylabel("Density", fontsize=FONT_SIZE_LABELS, fontweight='bold', fontproperties=arial_font)
axes[-1].legend(prop=fm.FontProperties(fname=ARIAL, size=FONT_SIZE_LEGEND), frameon=False, loc='upper right')

plt.tight_layout()
plt.savefig("S2A_score_distributions_by_radius_2026-10.svg", dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
#@title S2B: neighborhood size and residues with no observed missense variation, by radius
# S2_radius_summary_2026-10.csv: per radius, the mean / median number
# of other residues within the sphere (num_contacts) and the share of residues whose neighborhood (focal residue included)
# has no observed gnomAD missense variant (cs_mis_obs = 0).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pypalettes import load_cmap

FIGURE_SIZE = (6, 6)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 18
FONT_SIZE_TEXT = 15
SPINE_WIDTH = 1.2

arial_font = fm.FontProperties(fname=ARIAL)
colors = load_cmap('Royal1').colors

summ = pd.read_csv("S2_radius_summary_2026-10.csv")
print(summ[['radius', 'proteins', 'residues', 'contacts_mean', 'contacts_median', 'zero_obs_pct']].to_string(index=False))
x = np.arange(len(summ))

fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)
ax.set_box_aspect(1)
ax.bar(x, summ.contacts_mean, width=0.6, color=colors[0], edgecolor='black', linewidth=SPINE_WIDTH)
ax2 = ax.twinx()
ax2.plot(x, summ.zero_obs_pct, marker='o', markersize=9, linewidth=2.5, color=colors[1], markeredgecolor='black')
for xi, v in zip(x, summ.zero_obs_pct):
    ax2.text(xi + 0.1, v + 1.5, f"{v:.1f}%", ha='left', va='bottom', fontproperties=arial_font, fontsize=FONT_SIZE_TEXT, color=colors[1],
              bbox=dict(facecolor='white', edgecolor='none', pad=1))
ax.set_xticks(x)
ax.set_xticklabels([f"{r} Å" for r in summ.radius])
ax.set_xlabel("Neighborhood radius", fontsize=FONT_SIZE_LABELS, fontweight='bold', fontproperties=arial_font)
ax.set_ylabel("Residues in neighborhood (mean)", fontsize=FONT_SIZE_LABELS, fontweight='bold', fontproperties=arial_font)
ax2.set_ylabel("No observed missense variant (%)", fontsize=FONT_SIZE_LABELS, fontweight='bold', fontproperties=arial_font,
               color=colors[1])
ax2.set_ylim(0, 55)
for a, side in [(ax, 'left'), (ax2, 'right')]:
    a.tick_params(axis='both', which='major', labelsize=FONT_SIZE_TICKS, width=SPINE_WIDTH)
    for label in a.get_xticklabels() + a.get_yticklabels():
        label.set_fontproperties(arial_font)
        label.set_fontsize(FONT_SIZE_TICKS)
    a.spines['top'].set_visible(False)
    for spine in ['left', 'bottom', 'right']:
        a.spines[spine].set_linewidth(SPINE_WIDTH)
        a.spines[spine].set_color('black')
ax2.tick_params(axis='y', colors=colors[1])
ax2.spines['right'].set_color(colors[1])

plt.tight_layout()
plt.savefig("S2B_neighborhood_size_by_radius_2026-10.svg", dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
#@title S2C: residues below the HSC cutoff (missense vs synonymous) and synonymous calibration, by radius
# S2_radius_summary_2026-10.csv: share of all residues with missense HuSC < -2.326 and with synonymous score < -2.326
# (same null and neighborhood); empirical FDR = synonymous share / missense share. syn_p05_pct = share of synonymous
# permutation p-values < 0.05 (5% if the null were exactly calibrated).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.ticker import FuncFormatter
from pypalettes import load_cmap

FIGURE_SIZE = (6, 6)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 18
FONT_SIZE_LEGEND = 15
FONT_SIZE_TEXT = 14
SPINE_WIDTH = 1.2

arial_font = fm.FontProperties(fname=ARIAL)
colors = load_cmap('Royal1').colors

summ = pd.read_csv("S2_radius_summary_2026-10.csv")
print(summ[['radius', 'mis_below_pct', 'syn_below_pct', 'emp_fdr', 'syn_p05_pct', 'syn_ks']].to_string(index=False))
x = np.arange(len(summ)); w = 0.38

fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)
ax.set_box_aspect(1)
ax.bar(x - w / 2, summ.mis_below_pct, width=w, color=colors[1], edgecolor='black', linewidth=SPINE_WIDTH, label='Missense (HuSC)')
ax.bar(x + w / 2, summ.syn_below_pct, width=w, color=colors[0], edgecolor='black', linewidth=SPINE_WIDTH, label='Synonymous')
for xi, (m, f) in enumerate(zip(summ.mis_below_pct, summ.emp_fdr)):
    ax.text(xi - w / 2, m * 1.25, f"{f:.1%}", ha='center', va='bottom', fontproperties=arial_font, fontsize=FONT_SIZE_TEXT,
            color=colors[1])
ax.text(0.97, 0.97, "Labels: empirical FDR", transform=ax.transAxes, ha='right', va='top', fontproperties=arial_font,
        fontsize=FONT_SIZE_TEXT, color=colors[1])
ax.set_yscale('log')
ax.set_ylim(0.005, 1000)
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
ax.set_xticks(x)
ax.set_xticklabels([f"{r} Å" for r in summ.radius])
ax.set_xlabel("Neighborhood radius", fontsize=FONT_SIZE_LABELS, fontweight='bold', fontproperties=arial_font)
ax.set_ylabel("Residues with score < −2.326 (%)", fontsize=FONT_SIZE_LABELS, fontweight='bold', fontproperties=arial_font)
ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE_TICKS, width=SPINE_WIDTH, bottom=True, left=True)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font)
    label.set_fontsize(FONT_SIZE_TICKS)
ax.legend(prop=fm.FontProperties(fname=ARIAL, size=FONT_SIZE_LEGEND), frameon=False, loc='upper left', bbox_to_anchor=(0, 0.9))
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(SPINE_WIDTH)
    ax.spines[spine].set_color('black')

plt.tight_layout()
plt.savefig("S2C_residues_below_cutoff_by_radius_2026-10.svg", dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
#@title S2D: ClinVar pathogenic residues vs background residues, paired change in AUROC relative to 8 Å
# S2_background_auroc_by_radius_2026-10.csv: 7,077 ClinVar pathogenic residues vs 105,575
# background residues (up to 100 random non-ClinVar residues per protein, default_rng(42)) from the same 1,057 proteins; labels do
# not use allele frequency. AUROC of -HuSC over all residues, and structure-matched AUROC (within strata of 8 Å neighborhood-size
# quartile x pLDDT bin <50/50-70/70-90/>90, weighted by pathogenic count; strata with >= 10 pathogenic). ΔAUROC vs 8 Å with 95%
# CIs from 1,000 paired protein bootstrap draws.
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from pypalettes import load_cmap

FIGURE_SIZE = (6, 6)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 18
FONT_SIZE_TEXT = 14
SPINE_WIDTH = 1.2

arial_font = fm.FontProperties(fname=ARIAL)
colors = load_cmap('Royal1').colors
SETS = {'Pathogenic vs background': ('All residues', colors[0]),
        'Pathogenic vs background (structure-matched)': ('Structure-matched', colors[1])}

path = "S2_background_auroc_by_radius_2026-10.csv"
auc = pd.read_csv(path)
print(auc[['set', 'radius', 'auroc', 'lo', 'hi', 'diff_vs8', 'diff_lo', 'diff_hi', 'p_vs8']].to_string(index=False, float_format=lambda v: f"{v:.3f}"))
radii = sorted(auc.radius.unique())

fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)
ax.set_box_aspect(1)
ax.axhline(0, color='black', linestyle='--', linewidth=1, zorder=0)
for k, (s, (label, color)) in enumerate(SETS.items()):
    d = auc[auc.set == s].sort_values('radius')
    off = (k - 0.5) * 0.3
    ax.errorbar(d.radius + off, d.diff_vs8, yerr=[d.diff_vs8 - d.diff_lo, d.diff_hi - d.diff_vs8], fmt='o-', color=color, ecolor=color,
                markersize=8, linewidth=2.5, capsize=4, markeredgecolor='black', markeredgewidth=0.6)
    ax.text(0.97, 0.03 + 0.07 * (1 - k), label, transform=ax.transAxes, ha='right', va='bottom', fontproperties=arial_font,
            fontsize=FONT_SIZE_TEXT, color=color)
n = auc.iloc[0]
ax.text(0.97, 0.17, f"{int(n.n_path):,} pathogenic / {int(n.n_ben):,} background", transform=ax.transAxes, ha='right', va='bottom',
        fontproperties=arial_font, fontsize=FONT_SIZE_TEXT, color='black')
ax.set_xticks(radii)
ax.set_xlim(3.5, 15.5)
ax.set_xlabel("Neighborhood radius (Å)", fontsize=FONT_SIZE_LABELS, fontweight='bold', fontproperties=arial_font)
ax.set_ylabel("Δ ROC AUC vs. 8 Å", fontsize=FONT_SIZE_LABELS, fontweight='bold', fontproperties=arial_font)
ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE_TICKS, width=SPINE_WIDTH, bottom=True, left=True)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font)
    label.set_fontsize(FONT_SIZE_TICKS)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(SPINE_WIDTH)
    ax.spines[spine].set_color('black')

plt.tight_layout()
plt.savefig("S2D_background_delta_auroc_by_radius_2026-10.svg", dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
#@title S3 setup: Load ESM2-650M for ClinVar position-level comparison
!pip install -q transformers accelerate

import torch
from transformers import AutoTokenizer, EsmForMaskedLM

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type != "cuda":
    print("WARNING: no GPU detected - switch runtime to GPU (Runtime > Change runtime type) before scoring 13k positions.")

MODEL_NAME = "facebook/esm2_t33_650M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = EsmForMaskedLM.from_pretrained(MODEL_NAME).to(device).eval()
print("Mask token:", tokenizer.mask_token, tokenizer.mask_token_id)
print("Model loaded.")

In [ ]:
#@title S3 setup: load ClinVar table + reference sequences, build worklist
# context_clinvar_with_HuSC_2026-10.csv (same ClinVar set as Fig 3B: 7,077 pathogenic / 7,596 benign, 3,016 proteins).
import pandas as pd
import numpy as np

def load_fasta(path):
    seqs = {}
    cur_id, cur_seq = None, []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith(">"):
                if cur_id:
                    seqs[cur_id] = "".join(cur_seq)
                header = line[1:]
                cur_id = header.split("|")[1] if "|" in header else header.split()[0]
                cur_seq = []
            else:
                cur_seq.append(line)
        if cur_id:
            seqs[cur_id] = "".join(cur_seq)
    return seqs

sequences = load_fasta("UniProt_Sequences.fasta")
print(f"Loaded {len(sequences)} reference sequences")

df = pd.read_csv("context_clinvar_with_HuSC_2026-10.csv")
method_cols = ["HuSC", "consurf", "phylop", "cosmis", "gerp", "phastcons", "mtr3d"]
df = df.dropna(subset=method_cols + ["label"]).copy()
print(f"Rows after matching main-text filter: {len(df)}")
print("Pathogenic:", (df['label'] == 1).sum(), "Benign:", (df['label'] == 0).sum())

def get_wt(row):
    seq = sequences.get(row["uniprot_id"])
    if seq is None:
        return None
    pos = int(row["uniprot_pos"])
    if pos < 1 or pos > len(seq):
        return None
    return seq[pos - 1]

df["ref_aa_check"] = df.apply(get_wt, axis=1)
n_missing_seq = df["ref_aa_check"].isna().sum()
mismatch = (df["ref_aa_check"].notna()) & (df["ref_aa_check"] != df["uniprot_aa"])
print(f"Missing reference sequence: {n_missing_seq}")
print(f"WT amino acid mismatches vs uniprot_aa column: {mismatch.sum()}")

df_scoreable = df[df["ref_aa_check"].notna() & ~mismatch].copy()
print(f"Rows scoreable with ESM2: {len(df_scoreable)}")

In [ ]:
#@title S3 setup: masked-marginal average LLR scoring per ClinVar position
import os
import torch

AMINO_ACIDS = list("ACDEFGHIKLMNPQRSTVWY")
aa_token_ids = tokenizer.convert_tokens_to_ids(AMINO_ACIDS)
aa_index = {aa: i for i, aa in enumerate(AMINO_ACIDS)}
aa_token_ids_t = torch.tensor(aa_token_ids, device=device)

CHECKPOINT_PATH = "esm2_650M_clinvar_scores.csv"
MAX_CTX = 1022
BATCH_SIZE = 16

if os.path.exists(CHECKPOINT_PATH):
    done_df = pd.read_csv(CHECKPOINT_PATH)
    done_keys = set(zip(done_df["uniprot_id"], done_df["uniprot_pos"]))
    results = done_df.to_dict("records")
    print(f"Resuming: {len(done_keys)} positions already scored")
else:
    done_keys = set()
    results = []

grouped = df_scoreable.groupby("uniprot_id")
n_proteins = len(grouped)

with torch.no_grad():
    for pi, (uid, group) in enumerate(grouped):
        seq = sequences[uid]
        positions = [int(p) for p in group["uniprot_pos"].tolist()]
        wt_aas = group["uniprot_aa"].tolist()
        todo = [(p, aa) for p, aa in zip(positions, wt_aas) if (uid, p) not in done_keys]
        if not todo:
            continue

        for i in range(0, len(todo), BATCH_SIZE):
            chunk = todo[i:i + BATCH_SIZE]
            batch_seqs = []
            mask_indices = []
            for pos, wt_aa in chunk:
                if len(seq) <= MAX_CTX:
                    window_seq = seq
                    window_pos = pos - 1
                else:
                    half = MAX_CTX // 2
                    start = max(0, pos - 1 - half)
                    end = min(len(seq), start + MAX_CTX)
                    start = max(0, end - MAX_CTX)
                    window_seq = seq[start:end]
                    window_pos = (pos - 1) - start
                seq_list = list(window_seq)
                seq_list[window_pos] = tokenizer.mask_token
                batch_seqs.append("".join(seq_list))
                mask_indices.append(window_pos)

            enc = tokenizer(batch_seqs, return_tensors="pt", padding=True).to(device)
            out = model(**enc).logits
            log_probs = torch.log_softmax(out, dim=-1)

            for j, (pos, wt_aa) in enumerate(chunk):
                tok_pos = mask_indices[j] + 1  # +1 for prepended <cls>
                logp20 = log_probs[j, tok_pos, aa_token_ids_t]  # (20,)
                wt_i = aa_index[wt_aa]
                logp_wt = logp20[wt_i]
                mean_logp_alt = (logp20.sum() - logp_wt) / (len(AMINO_ACIDS) - 1)
                avg_neg_llr = (logp_wt - mean_logp_alt).item()
                results.append({
                    "uniprot_id": uid,
                    "uniprot_pos": pos,
                    "uniprot_aa": wt_aa,
                    "esm2_avg_neg_llr": avg_neg_llr,
                })

        if pi % 25 == 0 or pi == n_proteins - 1:
            pd.DataFrame(results).to_csv(CHECKPOINT_PATH, index=False)
            print(f"[{pi+1}/{n_proteins}] proteins done, {len(results)} positions scored")

pd.DataFrame(results).to_csv(CHECKPOINT_PATH, index=False)
print(f"Done. Total scored: {len(results)}")

In [ ]:
#@title S3 setup: merge esm2_avg_neg_llr into main ClinVar table
# writes context_clinvar_with_HuSC_esm2_2026-10.csv (Fig 3B ClinVar set + ESM2-650M masked-marginal score).
esm2_df = pd.read_csv(CHECKPOINT_PATH)

# sanity check: duplicate-key scores should be identical (position-level score, substitution-independent)
dup_check = esm2_df[esm2_df.duplicated(subset=["uniprot_id", "uniprot_pos"], keep=False)]
if len(dup_check) > 0:
    spread = dup_check.groupby(["uniprot_id", "uniprot_pos"])["esm2_avg_neg_llr"].agg(lambda s: s.max() - s.min())
    print(f"Duplicate-key rows: {len(dup_check)}; max within-key score spread: {spread.max():.2e}")

esm2_df = esm2_df.drop_duplicates(subset=["uniprot_id", "uniprot_pos"], keep="first")

merged = df.merge(
    esm2_df[["uniprot_id", "uniprot_pos", "esm2_avg_neg_llr"]],
    on=["uniprot_id", "uniprot_pos"],
    how="left",
)
merged.to_csv("context_clinvar_with_HuSC_esm2_2026-10.csv", index=False)
print("Saved context_clinvar_with_HuSC_esm2_2026-10.csv")
print(f"Rows: {len(merged)} (expected {len(df)})")
print(f"Rows with ESM2 score: {merged['esm2_avg_neg_llr'].notna().sum()} / {len(merged)}")
print(merged["esm2_avg_neg_llr"].describe())

In [ ]:
#@title S3 data: ROC/PR AUC with protein bootstrap for all methods (incl. ESM2-650M)
# context_clinvar_with_HuSC_esm2_2026-10.csv = Fig 3B ClinVar set + ESM2-650M masked-marginal average LLR.
# Same protein bootstrap as Fig 3B (proteins resampled with replacement, default_rng(42), 1,000 draws, shared across
# methods); PR AUC = average precision, as in the main text.
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

df_esm = pd.read_csv("context_clinvar_with_HuSC_esm2_2026-10.csv")

column_labels = {
    'HuSC': 'HuSC',
    'consurf': 'ConSurf',
    'phylop': 'phyloP',
    'cosmis': 'COSMIS',
    'gerp': 'GERP++',
    'phastcons': 'phastCons',
    'mtr3d': 'MTR3D',
    'esm2_avg_neg_llr': 'ESM2-650M',
}
columns_to_plot = list(column_labels.keys())
label_column = 'label'
N_BOOT = 1000

df_esm = df_esm.dropna(subset=columns_to_plot + [label_column]).reset_index(drop=True)
print(f"n = {len(df_esm)} (pathogenic: {(df_esm[label_column]==1).sum()}, benign: {(df_esm[label_column]==0).sum()}, "
      f"proteins: {df_esm.uniprot_id.nunique()})")

# invert scores that are lower-for-more-conserved (matches main-text convention)
for col in ['HuSC', 'consurf', 'cosmis', 'mtr3d']:
    df_esm[col] = -df_esm[col]
# esm2_avg_neg_llr and gerp/phylop/phastcons already higher = more conserved/pathogenic-associated

y_true = df_esm[label_column].values
groups = [g.index.values for _, g in df_esm.groupby('uniprot_id')]
rng = np.random.default_rng(42)
draws = []
for _ in range(N_BOOT):
    idx = np.concatenate([groups[k] for k in rng.integers(0, len(groups), len(groups))])
    if len(np.unique(y_true[idx])) == 2:
        draws.append(idx)

roc_point = {c: roc_auc_score(y_true, df_esm[c].values) for c in columns_to_plot}
pr_point = {c: average_precision_score(y_true, df_esm[c].values) for c in columns_to_plot}
roc_scores = {c: np.array([roc_auc_score(y_true[i], df_esm[c].values[i]) for i in draws]) for c in columns_to_plot}
pr_scores = {c: np.array([average_precision_score(y_true[i], df_esm[c].values[i]) for i in draws]) for c in columns_to_plot}

auc_table = pd.DataFrame([
    dict(method=column_labels[c],
         roc_auc=roc_point[c], roc_lo=np.percentile(roc_scores[c], 2.5), roc_hi=np.percentile(roc_scores[c], 97.5),
         pr_auc=pr_point[c], pr_lo=np.percentile(pr_scores[c], 2.5), pr_hi=np.percentile(pr_scores[c], 97.5))
    for c in columns_to_plot]).sort_values('roc_auc', ascending=False)
auc_table.to_csv("S3_clinvar_auc_esm2_2026-10.csv", index=False)
print(f"\n=== ROC / PR AUC (95% protein bootstrap CI, {len(draws)} resamples) ===")
print(auc_table.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

In [ ]:
#@title S3 stats: paired protein-bootstrap significance (HuSC vs all, ROC & PR AUC)
# Difference in each shared resample; empirical two-sided p floored at 1/n_resamples; BH-FDR over the 7 comparisons.
from statsmodels.stats.multitest import multipletests

ref = 'HuSC'
competitors = [c for c in columns_to_plot if c != ref]

for label, scores_dict, point in [("ROC AUC", roc_scores, roc_point), ("PR AUC", pr_scores, pr_point)]:
    rows = []
    for col in competitors:
        diff = scores_dict[ref] - scores_dict[col]
        p = min(2 * min((diff <= 0).mean(), (diff >= 0).mean()), 1.0)
        rows.append((col, point[ref] - point[col], *np.percentile(diff, [2.5, 97.5]), max(p, 1 / len(diff))))
    qvals = multipletests([r[4] for r in rows], method='fdr_bh')[1]
    print(f"\n=== HuSC vs. competitors: {label} paired protein-bootstrap differences ===")
    for (col, d, lo, hi, p), q in zip(rows, qvals):
        print(f"HuSC - {column_labels[col]:10s} Δ = {d:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  q = {q:.3g}")

In [ ]:
#@title S3A: ROC and PR AUC grouped bar plot (incl. ESM2-650M)
# Uses auc_table from "S3 data" (point AUCs, 95% protein-bootstrap CIs); methods ordered by ROC AUC.
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
from pypalettes import load_cmap

FIGURE_SIZE = (6, 6)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 18
FONT_SIZE_LEGEND = 15
SPINE_WIDTH = 1.2

arial_font = fm.FontProperties(fname=ARIAL)
colors = load_cmap('Royal1').colors
METRICS = [('roc', 'ROC AUC', colors[0]), ('pr', 'PR AUC', colors[3])]

fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)
ax.set_box_aspect(1)

x = np.arange(len(auc_table))
width = 0.38
for k, (m, label, color) in enumerate(METRICS):
    vals = auc_table[f'{m}_auc'].values
    offs = x + (k - 0.5) * width
    ax.bar(offs, vals - 0.4, bottom=0.4, width=width, color=color, label=label)
    ax.errorbar(offs, vals, yerr=[vals - auc_table[f'{m}_lo'].values, auc_table[f'{m}_hi'].values - vals],
                fmt='none', ecolor='black', capsize=3, linewidth=1.2)

ax.set_xticks(x)
ax.set_xticklabels(auc_table['method'], ha='right', rotation=45, rotation_mode='anchor')
ax.set_ylabel("AUC", fontsize=FONT_SIZE_LABELS, fontweight='bold', fontproperties=arial_font)
ax.set_ylim(0.4, 1)

ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE_TICKS, width=SPINE_WIDTH, bottom=True, left=True)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font)
    label.set_fontsize(FONT_SIZE_TICKS)

ax.legend(prop=fm.FontProperties(fname=ARIAL, size=FONT_SIZE_LEGEND), frameon=False, loc='upper right')

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(SPINE_WIDTH)
    ax.spines[spine].set_color('black')

plt.tight_layout()
plt.savefig("S3A_roc_pr_auc_barplot_2026-10.svg", dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
#@title S3B: frequency-matched ClinVar AUROC by inheritance, incl. ESM2-650M
# clinvar_inheritance_2026-10.tsv (ClinVar variants with their gnomAD AF and the gene's inheritance mode) joined with the ESM2-650M scores from
# context_clinvar_with_HuSC_esm2_2026-10.csv. Frequency-matched = variant AF < 1e-4 in both classes. CIs: 95% protein
# bootstrap (default_rng(42) per group, 1,000 draws, shared across methods); paired HuSC - other differences, empirical
# two-sided p floored at 1/1000, BH-FDR across all 10 comparisons (5 per group), as in the main text.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score
from statsmodels.stats.multitest import multipletests
from pypalettes import load_cmap
from matplotlib import font_manager as fm

# --- Configuration ---
FIGURE_SIZE = (6, 6)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 18
FONT_SIZE_LEGEND = 15
SPINE_WIDTH = 1.2
N_BOOT = 1000
AF_MAX = 1e-4

arial_font = fm.FontProperties(fname=ARIAL)
colors = load_cmap('Royal1').colors
GROUPS = {'dominant/HI': ('Dominant / HI', colors[1]), 'recessive': ('Recessive', colors[0])}

# Score columns; lower = more constrained / conserved for HuSC, ConSurf, COSMIS, MTR3D
fm_labels = {'HuSC': 'HuSC', 'consurf': 'ConSurf', 'phylop': 'phyloP', 'cosmis': 'COSMIS', 'mtr3d': 'MTR3D',
             'esm2_avg_neg_llr': 'ESM2-650M'}
sign = {'HuSC': -1, 'consurf': -1, 'phylop': 1, 'cosmis': -1, 'mtr3d': -1, 'esm2_avg_neg_llr': 1}

# --- Load data ---
inh = pd.read_csv('clinvar_inheritance_2026-10.tsv', sep='\t')
esm = pd.read_csv('context_clinvar_with_HuSC_esm2_2026-10.csv')[['uniprot_id', 'uniprot_pos', 'esm2_avg_neg_llr']]
esm = esm.drop_duplicates(['uniprot_id', 'uniprot_pos'])
fmd = inh.merge(esm, on=['uniprot_id', 'uniprot_pos'], how='left')
assert len(fmd) == len(inh)
fmd = fmd[(fmd['variant_af_max'] < AF_MAX) & fmd['inheritance'].isin(list(GROUPS))]
fmd = fmd.dropna(subset=list(fm_labels) + ['label']).reset_index(drop=True)

fm_results, fm_boot, rows = {}, {}, []
for g in GROUPS:
    d = fmd[fmd['inheritance'] == g].reset_index(drop=True)
    y = d['label'].values
    print(f"{GROUPS[g][0]}: {int((y == 1).sum())} pathogenic, {int((y == 0).sum())} benign, {d.uniprot_id.nunique()} proteins")
    groups = [gr.index.values for _, gr in d.groupby('uniprot_id')]
    rng = np.random.default_rng(42)
    draws = []
    for _ in range(N_BOOT):
        idx = np.concatenate([groups[k] for k in rng.integers(0, len(groups), len(groups))])
        if len(np.unique(y[idx])) == 2:
            draws.append(idx)
    for col in fm_labels:
        s = sign[col] * d[col].values
        boots = np.array([roc_auc_score(y[i], s[i]) for i in draws])
        fm_boot[(g, col)] = boots
        fm_results[(g, col)] = (roc_auc_score(y, s), *np.percentile(boots, [2.5, 97.5]))
        p, lo, hi = fm_results[(g, col)]
        print(f"  {fm_labels[col]:10s} {p:.3f} [{lo:.3f}, {hi:.3f}]")
    for col in [c for c in fm_labels if c != 'HuSC']:
        diff = fm_boot[(g, 'HuSC')] - fm_boot[(g, col)]
        p = max(min(2 * min((diff <= 0).mean(), (diff >= 0).mean()), 1.0), 1 / len(diff))
        rows.append((g, col, fm_results[(g, 'HuSC')][0] - fm_results[(g, col)][0], *np.percentile(diff, [2.5, 97.5]), p))

qs = multipletests([r[5] for r in rows], method='fdr_bh')[1]
print("\nHuSC - other (paired protein bootstrap; BH across all 10 comparisons)")
for (g, col, dd, lo, hi, p), q in zip(rows, qs):
    print(f"  {GROUPS[g][0]:13s} HuSC - {fm_labels[col]:10s} Δ = {dd:+.3f} [{lo:+.3f}, {hi:+.3f}]  p = {p:.3g}  q = {q:.3g}")

# --- Create figure ---
fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)
ax.set_box_aspect(1)

x = np.arange(len(fm_labels))
width = 0.38
for k, (g, (label, color)) in enumerate(GROUPS.items()):
    vals = np.array([fm_results[(g, c)] for c in fm_labels])
    offs = x + (k - 0.5) * width
    ax.bar(offs, vals[:, 0] - 0.4, bottom=0.4, width=width, color=color, label=label)
    ax.errorbar(offs, vals[:, 0], yerr=[vals[:, 0] - vals[:, 1], vals[:, 2] - vals[:, 0]],
                fmt='none', ecolor='black', capsize=3, linewidth=1.2)

ax.axhline(0.5, color='black', linestyle='--', linewidth=SPINE_WIDTH)
ax.set_xticks(x)
ax.set_xticklabels(list(fm_labels.values()), ha='right', rotation=45, rotation_mode='anchor')
ax.set_ylabel("ROC AUC (variant AF < 10$^{-4}$)", fontsize=FONT_SIZE_LABELS,
              fontweight='bold', fontproperties=arial_font)
ax.set_ylim(0.4, 1)

ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE_TICKS, width=SPINE_WIDTH, bottom=True, left=True)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font)
    label.set_fontsize(FONT_SIZE_TICKS)

ax.legend(prop=fm.FontProperties(fname=ARIAL, size=FONT_SIZE_LEGEND), frameon=False, loc='upper right')

for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(SPINE_WIDTH)
    ax.spines[spine].set_color('black')

plt.tight_layout()
plt.savefig("S3B_roc_auc_inheritance_freqmatched_esm2_2026-10.svg", dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
#@title S3C: ROC curves (incl. ESM2-650M and HuSC at other neighborhood radii)
# Uses df_esm / roc_point from "S3 data"; legend ordered by ROC AUC.
# HuSC at 5, 11 and 14 Å: per-site scores from S2_clinvar_by_radius_2026-10.csv (same variants; the 8 Å column equals HuSC).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score
from matplotlib import font_manager as fm

FIGURE_SIZE = (6, 6)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 18
FONT_SIZE_LEGEND = 11
SPINE_WIDTH = 1.2
LINE_WIDTH = 2

arial_font = fm.FontProperties(fname=ARIAL)
# Royal1 / AsteroidCity2 colors; HuSC in the main-figure red, population metrics (COSMIS, MTR3D) dashed
CURVE_STYLE = {
    'HuSC': ('#C93312', '-', 3.0), 'consurf': ('#DC863B', '-', LINE_WIDTH), 'phylop': ('#899DA4', '-', LINE_WIDTH),
    'esm2_avg_neg_llr': ('black', '-', LINE_WIDTH), 'gerp': ('#AC9765', '-', LINE_WIDTH),
    'phastcons': ('#175149', '-', LINE_WIDTH), 'cosmis': ('#54D8B1', '--', LINE_WIDTH), 'mtr3d': ('#B67C3B', '--', LINE_WIDTH),
    'HuSC_5': ('#F1A38E', '-.', 1.6), 'HuSC_11': ('#E2694B', '-.', 1.6), 'HuSC_14': ('#7A1F0B', '-.', 1.6),
}
RADIUS_LABELS = {'HuSC_5': 'HuSC 5 Å', 'HuSC_11': 'HuSC 11 Å', 'HuSC_14': 'HuSC 14 Å'}
curve_labels = {**column_labels, 'HuSC': 'HuSC 8 Å', **RADIUS_LABELS}

# Add HuSC at the other radii (lower = more constrained, so negate as for HuSC in "S3 data")
rad = pd.read_csv("S2_clinvar_by_radius_2026-10.csv")[['uniprot_id', 'uniprot_pos', *RADIUS_LABELS]]
df_rad = df_esm.merge(rad, on=['uniprot_id', 'uniprot_pos'], how='left')
assert len(df_rad) == len(df_esm) and df_rad[list(RADIUS_LABELS)].notna().all().all()
for c in RADIUS_LABELS:
    df_rad[c] = -df_rad[c]

y_true_plot = df_rad[label_column].values
roc_all = {**roc_point, **{c: roc_auc_score(y_true_plot, df_rad[c].values) for c in RADIUS_LABELS}}
print({curve_labels[c]: round(v, 3) for c, v in sorted(roc_all.items(), key=lambda kv: -kv[1])})

fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)
for col in sorted(roc_all, key=lambda c: -roc_all[c]):
    fpr, tpr, _ = roc_curve(y_true_plot, df_rad[col].values)
    color, ls, lw = CURVE_STYLE[col]
    ax.plot(fpr, tpr, color=color, linestyle=ls, linewidth=lw,
            label=f'{curve_labels[col]} ({roc_all[col]:.2f})')
ax.plot([0, 1], [0, 1], color='black', linestyle=':', linewidth=SPINE_WIDTH)
ax.set_xlabel('False positive rate', fontsize=FONT_SIZE_LABELS, fontweight='bold', fontproperties=arial_font)
ax.set_ylabel('True positive rate', fontsize=FONT_SIZE_LABELS, fontweight='bold', fontproperties=arial_font)
ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.0]); ax.set_aspect('equal')
ax.legend(loc='lower right', frameon=False, prop=fm.FontProperties(fname=ARIAL, size=FONT_SIZE_LEGEND),
          handlelength=1.8, borderaxespad=0.2, labelspacing=0.3)
ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE_TICKS, width=SPINE_WIDTH, bottom=True, left=True)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font); label.set_fontsize(FONT_SIZE_TICKS)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(SPINE_WIDTH); ax.spines[spine].set_color('black')
plt.tight_layout()
plt.savefig("S3C_roc_curves_2026-10.svg", dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
#@title S3D: precision-recall curves (incl. ESM2-650M and HuSC at other neighborhood radii)
# Uses df_esm / pr_point from "S3 data" and df_rad / curve_labels / CURVE_STYLE / RADIUS_LABELS from S3C (run S3C first).
# PR AUC = average precision, so curves are drawn as steps. Legend ordered by PR AUC.
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, average_precision_score
from matplotlib import font_manager as fm

FIGURE_SIZE = (6, 6)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 18
FONT_SIZE_LEGEND = 11
SPINE_WIDTH = 1.2

arial_font = fm.FontProperties(fname=ARIAL)

y_true_plot = df_rad[label_column].values
baseline = y_true_plot.mean()
pr_all = {**pr_point, **{c: average_precision_score(y_true_plot, df_rad[c].values) for c in RADIUS_LABELS}}
print({curve_labels[c]: round(v, 3) for c, v in sorted(pr_all.items(), key=lambda kv: -kv[1])})

fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)
for col in sorted(pr_all, key=lambda c: -pr_all[c]):
    precision, recall, _ = precision_recall_curve(y_true_plot, df_rad[col].values)
    color, ls, lw = CURVE_STYLE[col]
    ax.step(recall, precision, where='post', color=color, linestyle=ls, linewidth=lw,
            label=f'{curve_labels[col]} ({pr_all[col]:.2f})')
ax.axhline(y=baseline, color='black', linestyle=':', linewidth=SPINE_WIDTH)
ax.set_xlabel('Recall', fontsize=FONT_SIZE_LABELS, fontweight='bold', fontproperties=arial_font)
ax.set_ylabel('Precision', fontsize=FONT_SIZE_LABELS, fontweight='bold', fontproperties=arial_font)
ax.set_xlim([0.0, 1.0]); ax.set_ylim([0.0, 1.0]); ax.set_aspect('equal')
ax.legend(loc='lower left', frameon=False, prop=fm.FontProperties(fname=ARIAL, size=FONT_SIZE_LEGEND),
          handlelength=1.8, borderaxespad=0.2, labelspacing=0.3)
ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE_TICKS, width=SPINE_WIDTH, bottom=True, left=True)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font); label.set_fontsize(FONT_SIZE_TICKS)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(SPINE_WIDTH); ax.spines[spine].set_color('black')
plt.tight_layout()
plt.savefig("S3D_pr_curves_2026-10.svg", dpi=DPI, bbox_inches='tight')
plt.show()
print(f"Baseline (pathogenic fraction) = {baseline:.3f}")

In [ ]:
#@title S4A: residues called and empirical FDR across HuSC cutoffs (ConSurf > 3 residues)
# hsc_threshold_inputs_2026-10.tsv.gz: every residue of all_conservation_scores_with_HuSC_2026-10.csv with HuSC, synonymous score
# (syn_z = Phi^-1 of the synonymous permutation p, same null), ConSurf, gene name and relative local gnomAD coverage (rel_AN =
# median AN within +-15 codons / transcript median AN). Reproduces Fig 4 (926 residues, 469 genes). Set: ConSurf > 3 residues, excluding low coverage (rel_AN < 0.5), as for the HSC residues in Fig 4.
# Empirical FDR(t) = fraction of residues with syn_z < t / fraction with HuSC < t (synonymous variation as the null).
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.ticker import FuncFormatter
from pypalettes import load_cmap

FIGURE_SIZE = (6, 6)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 18
FONT_SIZE_LEGEND = 15
FONT_SIZE_TEXT = 13
SPINE_WIDTH = 1.2
CUTOFFS = [-1.282, -1.645, -1.960, -2.326, -2.576, -3.090]   # one-sided p = 0.1, 0.05, 0.025, 0.01, 0.005, 0.001
HSC_CUTOFF = -2.326

arial_font = fm.FontProperties(fname=ARIAL)
colors = load_cmap('Royal1').colors

path = "hsc_threshold_inputs_2026-10.tsv.gz"
res = pd.read_csv(path, sep='\t')
elig = res[(res.consurf > 3) & ~(res.rel_AN < 0.5)]
n = len(elig)
print(f"ConSurf > 3 residues: {(res.consurf > 3).sum():,}; after coverage filter: {n:,}")

rows = []
for t in CUTOFFS:
    n_mis, n_syn = int((elig.HuSC < t).sum()), int((elig.syn_z < t).sum())
    rows.append(dict(cutoff=t, n_missense=n_mis, n_synonymous=n_syn, pct_missense=100 * n_mis / n, pct_synonymous=100 * n_syn / n,
                     fdr=n_syn / n_mis, n_genes=elig.loc[elig.HuSC < t, 'gene_name'].nunique()))
fdr_tab = pd.DataFrame(rows)
fdr_tab.to_csv("S4A_empirical_fdr_2026-10.csv", index=False)
print(fdr_tab.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)
ax.set_box_aspect(1)
ax.plot(fdr_tab.cutoff, fdr_tab.pct_missense, marker='o', markersize=8, linewidth=2.5, color=colors[1], label='Missense (HuSC)')
ax.plot(fdr_tab.cutoff, fdr_tab.pct_synonymous, marker='o', markersize=8, linewidth=2.5, color=colors[0], label='Synonymous')
for r in fdr_tab.itertuples():           # empirical FDR above each missense point
    ax.text(r.cutoff, r.pct_missense * 1.45, f"{r.fdr:.0%}", ha='center', va='bottom', fontproperties=arial_font,
            fontsize=FONT_SIZE_TEXT, color=colors[1])
ax.text(0.97, 0.03, "Labels: empirical FDR", transform=ax.transAxes, ha='right', va='bottom', fontproperties=arial_font,
        fontsize=FONT_SIZE_TEXT, color=colors[1])
ax.axvline(HSC_CUTOFF, color='black', linestyle='--', linewidth=SPINE_WIDTH, zorder=0)
ax.set_yscale('log')
ax.set_xlim(-3.3, -1.1)
ax.set_ylim(0.01, 100)
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}"))
ax.set_xticks(CUTOFFS)
ax.set_xticklabels([f"{t:.2f}".replace('-', '−') for t in CUTOFFS], rotation=45, ha='right', rotation_mode='anchor')
ax.set_xlabel("Score cutoff", fontsize=FONT_SIZE_LABELS, fontweight='bold', fontproperties=arial_font)
ax.set_ylabel("Residues below cutoff (%)", fontsize=FONT_SIZE_LABELS, fontweight='bold', fontproperties=arial_font)
ax.tick_params(axis='both', which='major', labelsize=FONT_SIZE_TICKS, width=SPINE_WIDTH, bottom=True, left=True)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font)
    label.set_fontsize(FONT_SIZE_TICKS)
leg = ax.legend(prop=fm.FontProperties(fname=ARIAL, size=FONT_SIZE_LEGEND), frameon=True, loc='upper left',
                fancybox=False, edgecolor='none', framealpha=1)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(SPINE_WIDTH)
    ax.spines[spine].set_color('black')

plt.tight_layout()
plt.savefig("S4A_empirical_fdr_2026-10.svg", dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
#@title S4B setup: GO enrichment of the six Fig 4B terms across HuSC and ConSurf thresholds
# Same test as Fig 4B (Enrichr GO_Biological_Process_2025 passed by name, gene list = genes with >= 1 HSC residue, background =
# genes with >= 1 residue above the ConSurf cutoff, BH FDR), repeated while varying one threshold at a time:
#   HuSC cutoff -1.645 ... -3.090 at ConSurf > 3, and ConSurf cutoff at the 90th-99th percentile at HuSC < -2.326.
# HSC residues with low gnomAD coverage (rel_AN < 0.5) are excluded, as in Fig 4. Inputs: hsc_threshold_inputs_2026-10.tsv.gz.
!pip install -q gseapy

import json, time
import requests
import numpy as np
import pandas as pd
import gseapy as gp

requests.models.complexjson = json      # Enrichr returns Infinity odds ratios; let requests parse them (see Fig 4B cell)

GENE_SET_LIB = "GO_Biological_Process_2025"
FIG4B_TERMS = [
    "Regulation of Transcription by RNA Polymerase II (GO:0006357)",
    "Regulation of DNA-templated Transcription (GO:0006355)",
    "Defense Response to Virus (GO:0051607)",
    "Positive Regulation of Cytokine Production (GO:0001819)",
    "Regulation of Viral Genome Replication (GO:0045069)",
    "Cytokine-Mediated Signaling Pathway (GO:0019221)",
]
HUSC_CUTOFFS = [-1.645, -1.960, -2.326, -2.576, -3.090]
CONSURF_PCTS = [90, 95, 97.5, 99]
MAIN_HUSC, MAIN_CONSURF = -2.326, 3.0

res = pd.read_csv("hsc_threshold_inputs_2026-10.tsv.gz", sep='\t')
settings = [dict(axis='HuSC', label=f"{t:.2f}".replace('-', '−'), husc=t, consurf=MAIN_CONSURF) for t in HUSC_CUTOFFS]
settings += [dict(axis='ConSurf', label=f"{p:g}th", husc=MAIN_HUSC, consurf=float(np.percentile(res.consurf, p)))
             for p in CONSURF_PCTS]

def enrichr_with_retry(genes, background, max_retries=6, base_delay=10):
    for attempt in range(max_retries):
        try:
            return gp.enrichr(gene_list=genes, gene_sets=[GENE_SET_LIB], background=background, outdir=None).results
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            print(f"    Enrichr call failed ({e}); retrying in {base_delay * (attempt + 1)}s")
            time.sleep(base_delay * (attempt + 1))

rows = []
for s in settings:
    hsc = res[(res.HuSC < s['husc']) & (res.consurf > s['consurf']) & ~(res.rel_AN < 0.5)]
    genes = sorted(hsc.gene_name.unique())
    background = sorted(res.loc[res.consurf > s['consurf'], 'gene_name'].unique())
    enr = enrichr_with_retry(genes, background)
    q = enr.set_index('Term')['Adjusted P-value']
    for term in FIG4B_TERMS:
        rows.append(dict(axis=s['axis'], setting=s['label'], husc_cutoff=s['husc'], consurf_cutoff=s['consurf'],
                         n_residues=len(hsc), n_genes=len(genes), n_background=len(background), term=term, q=q.get(term, 1.0)))
    print(f"{s['axis']:7s} {s['label']:>6s}: HuSC < {s['husc']:.3f}, ConSurf > {s['consurf']:.3f} -> {len(hsc):,} residues, "
          f"{len(genes):,} genes, background {len(background):,}; " +
          ", ".join(f"{q.get(t, 1.0):.1e}" for t in FIG4B_TERMS))
    time.sleep(3)

go_grid = pd.DataFrame(rows)
go_grid.to_csv("S4B_go_threshold_grid_2026-10.csv", index=False)
print("\nSaved S4B_go_threshold_grid_2026-10.csv")

In [ ]:
#@title S4B: GO-term significance across HuSC and ConSurf thresholds (heatmap)
# Uses go_grid from "S4B setup" (S4B_go_threshold_grid_2026-10.csv). Color = -log10 BH-adjusted p (capped at 5, as in Fig 4B);
# cell text = adjusted p. Left block: HuSC cutoff varied at ConSurf > 3; right block: ConSurf percentile varied at HuSC < -2.326.
# The outlined column is the Fig 4 setting (HuSC < -2.326, ConSurf > 3).
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.patches import Rectangle

FIGURE_SIZE = (11, 6.5)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 15
FONT_SIZE_CELL = 11
SPINE_WIDTH = 1.2
COLOR_MAX = 5

arial_font = fm.FontProperties(fname=ARIAL)
TERM_LABELS = {
    "Regulation of Transcription by RNA Polymerase II (GO:0006357)": "Regulation of Transcription\nby RNA Polymerase II",
    "Regulation of DNA-templated Transcription (GO:0006355)": "Regulation of DNA-templated\nTranscription",
    "Defense Response to Virus (GO:0051607)": "Defense Response to Virus",
    "Positive Regulation of Cytokine Production (GO:0001819)": "Positive Regulation of\nCytokine Production",
    "Regulation of Viral Genome Replication (GO:0045069)": "Regulation of Viral\nGenome Replication",
    "Cytokine-Mediated Signaling Pathway (GO:0019221)": "Cytokine-Mediated\nSignaling Pathway",
}
grid = go_grid.copy()
cols = list(dict.fromkeys(zip(grid.axis, grid.setting)))           # setting order as computed
terms = list(TERM_LABELS)
q = np.array([[grid[(grid.axis == a) & (grid.setting == s) & (grid.term == t)].q.iloc[0] for (a, s) in cols] for t in terms])
gap = 0.6                                                           # visual gap between the two blocks
n_husc = sum(a == 'HuSC' for a, _ in cols)
xpos = np.array([i if i < n_husc else i + gap for i in range(len(cols))])

def fmt_q(v):
    if v >= 0.01:
        return f"{v:.2f}"
    m, e = f"{v:.0e}".split("e")
    return f"{m}×10$^{{{int(e)}}}$".replace("-", "−")

fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)
val = np.minimum(-np.log10(q), COLOR_MAX)
cmap = plt.get_cmap("Reds")
for i in range(len(terms)):
    for j, x in enumerate(xpos):
        ax.add_patch(Rectangle((x - 0.5, i - 0.5), 1, 1, facecolor=cmap(val[i, j] / COLOR_MAX), edgecolor='white', linewidth=1.5))
        ax.text(x, i, fmt_q(q[i, j]), ha='center', va='center', fontproperties=arial_font, fontsize=FONT_SIZE_CELL,
                color='white' if val[i, j] > 3 else 'black')
main_j = [j for j, (a, s) in enumerate(cols) if a == 'HuSC' and s == '−2.33'][0]
ax.add_patch(Rectangle((xpos[main_j] - 0.5, -0.5), 1, len(terms), fill=False, edgecolor='black', linewidth=2.5))

ax.set_xlim(xpos[0] - 0.5, xpos[-1] + 0.5)
ax.set_ylim(len(terms) - 0.5, -0.5)
ax.set_xticks(xpos)
ax.set_xticklabels([s for _, s in cols])
ax.set_yticks(range(len(terms)))
ax.set_yticklabels([TERM_LABELS[t] for t in terms])
ax.tick_params(axis='both', which='both', length=0, labelsize=FONT_SIZE_TICKS)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font)
    label.set_fontsize(FONT_SIZE_TICKS)
for spine in ax.spines.values():
    spine.set_visible(False)
# Block titles above the heatmap: which threshold varies, and the one held fixed
for x0, x1, title, sub in [(xpos[0], xpos[n_husc - 1], "HuSC cutoff", "ConSurf > 3"),
                           (xpos[n_husc], xpos[-1], "ConSurf percentile", "HuSC < −2.33")]:
    xm = (x0 + x1) / 2
    ax.text(xm, -0.95, title, ha='center', va='bottom', fontproperties=arial_font, fontsize=FONT_SIZE_LABELS, fontweight='bold')
    ax.text(xm, -0.62, f"({sub})", ha='center', va='bottom', fontproperties=arial_font, fontsize=FONT_SIZE_TICKS)
    ax.plot([x0 - 0.45, x1 + 0.45], [-0.55, -0.55], color='black', linewidth=SPINE_WIDTH, clip_on=False)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, COLOR_MAX))
cbar = plt.colorbar(sm, ax=ax, extend='max', fraction=0.03, pad=0.02)
cbar.set_label("−log$_{10}$(adjusted P)", fontsize=FONT_SIZE_LABELS, fontweight='bold', fontproperties=arial_font)
cbar.set_ticks(range(0, COLOR_MAX + 1))
cbar.set_ticklabels([str(t) for t in range(COLOR_MAX)] + [f"≥{COLOR_MAX}"])
cbar.ax.tick_params(labelsize=FONT_SIZE_TICKS, width=SPINE_WIDTH)
for label in cbar.ax.get_yticklabels():
    label.set_fontproperties(arial_font)
    label.set_fontsize(FONT_SIZE_TICKS)
cbar.outline.set_linewidth(SPINE_WIDTH)
plt.tight_layout()
plt.savefig("S4B_go_threshold_heatmap_2026-10.svg", dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
#@title S5A: ProteinGym-dev Spearman during fine-tuning (checkpoint selection), all model sizes
# finetune_history_esm2_{size}_husc.csv: ProteinGym dev Spearman (86 assays) every 500 steps; step 0 = base model
# (finetune_summary.csv base_pg_dev). Circles = checkpoint selected on the dev assays (used for all test results).
from pypalettes import load_cmap
_blues = load_cmap('Blues', cmap_type='continuous')
SIZE_COLORS = [_blues(v) for v in (0.4, 0.6, 0.8, 1.0)]
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

FIGURE_SIZE = (6, 6)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 15
SPINE_WIDTH = 1.2
SIZES = ['8M', '35M', '150M', '650M']
arial_font = fm.FontProperties(fname=ARIAL)

summ = pd.read_csv(f"finetune_summary.csv").set_index('run')
fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)
ax.set_box_aspect(1)
for size, color in zip(SIZES, SIZE_COLORS):
    run = f"esm2_{size}_husc"
    h = pd.read_csv(f"finetune_history_{run}.csv")
    x = np.r_[0, h.step.values]; y = np.r_[summ.at[run, 'base_pg_dev'], h.pg_dev.values]
    ax.plot(x, y, color=color, linestyle='-', linewidth=2.2)
    b = summ.at[run, 'best_step']
    ax.plot(b, h.set_index('step').pg_dev[b], marker='o', markersize=8, color=color,
            markerfacecolor=color, markeredgewidth=1.8, zorder=3)
    ax.text(x[-1] + 150, y[-1], size, color=color, va='center', fontproperties=arial_font, fontsize=FONT_SIZE_TICKS,
            fontweight='bold')
ax.set_xlim(0, 6700)
ax.set_xlabel("Training step", fontsize=FONT_SIZE_LABELS, fontproperties=arial_font)
ax.set_ylabel("Spearman correlation\n(ProteinGym dev, 86 assays)", fontsize=FONT_SIZE_LABELS, fontproperties=arial_font)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font); label.set_fontsize(FONT_SIZE_TICKS)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
for spine in ['left', 'bottom']:
    ax.spines[spine].set_linewidth(SPINE_WIDTH); ax.spines[spine].set_color('black')
ax.tick_params(which='both', bottom=True, left=True, width=SPINE_WIDTH, color='black', labelsize=FONT_SIZE_TICKS)
plt.tight_layout()
print(summ.loc[[f"esm2_{s}_husc" for s in SIZES], ['size', 'base_pg_dev', 'best_pg_dev', 'best_step', 'steps']].to_string())
plt.savefig("S5A_dev_spearman_training_2026-10.svg", dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
#@title S5B: Spearman gain from fine-tuning by taxon, all model sizes (ProteinGym test)
# Cells: median over assays of (fine-tuned - base) Spearman; stars: two-sided Wilcoxon signed-rank test of fine-tuned vs base
# within the category (* p < 0.05, ** p < 0.01, *** p < 0.001). Color scale shared by S5B (taxon) and S5C (selection type).
# Extends Fig 5C/D (650M) to all model sizes.
PANEL_GROUP = 'taxon'
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle
from scipy.stats import wilcoxon

DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 15
FONT_SIZE_CELL = 12
SPINE_WIDTH = 1.2
SIZES = ['8M', '35M', '150M', '650M']
GROUPS = {'taxon': ['Prokaryote', 'Eukaryote', 'Human', 'Virus'],
          'coarse_selection_type': ['Stability', 'Activity', 'OrganismalFitness', 'Expression', 'Binding']}
NICE = {'OrganismalFitness': 'Organismal fitness'}
arial_font = fm.FontProperties(fname=ARIAL)

pg = pd.read_csv(f"proteingym_scores_esm2_husc.csv")
pg = pg[pg.split == 'test'].reset_index(drop=True)

def stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''

rows = []
for col, cats in GROUPS.items():
    for c in cats:
        sub = pg[pg[col] == c]
        for s in SIZES:
            d = sub[f'ESM_{s}_ft_spearman'] - sub[f'ESM_{s}_base_spearman']
            rows.append(dict(group=col, category=c, n=len(sub), size=s, median_delta=d.median(),
                             p=wilcoxon(sub[f'ESM_{s}_ft_spearman'], sub[f'ESM_{s}_base_spearman']).pvalue))
tab = pd.DataFrame(rows)
vmax = np.abs(tab.median_delta).max()                        # shared by S5B and S5C
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
cmap = plt.get_cmap('coolwarm')
cats = GROUPS[PANEL_GROUP]
t = tab[tab.group == PANEL_GROUP]
print(t.pivot_table(index=['category', 'n'], columns='size', values='median_delta', sort=False)[SIZES].round(3).to_string())
print(t.pivot_table(index='category', columns='size', values='p', sort=False)[SIZES].to_string(float_format=lambda v: f"{v:.1e}"))

fig, ax = plt.subplots(figsize=(7.5, 1.1 * len(cats) + 1.4), dpi=DPI)
for i, c in enumerate(cats):
    for j, s in enumerate(SIZES):
        r = t[(t.category == c) & (t['size'] == s)].iloc[0]
        v = r.median_delta
        ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, facecolor=cmap(norm(v)), edgecolor='white', linewidth=1.5))
        txt = "0.00" if round(v, 2) == 0 else f"{v:+.2f}".replace('-', '−')
        ax.text(j, i, txt + stars(r.p), ha='center', va='center', fontproperties=arial_font, fontsize=FONT_SIZE_CELL,
                color='white' if abs(norm(v) - 0.5) > 0.35 else 'black')
ax.set_xlim(-0.5, len(SIZES) - 0.5)
ax.set_ylim(len(cats) - 0.5, -0.5)
ax.set_aspect('equal')
ax.set_xticks(range(len(SIZES))); ax.set_xticklabels(SIZES)
ax.xaxis.tick_top()
ax.set_xlabel("Model Size", fontsize=FONT_SIZE_LABELS, fontproperties=arial_font, labelpad=10)
ax.xaxis.set_label_position('top')
ax.set_yticks(range(len(cats)))
ax.set_yticklabels([f"{NICE.get(c, c)} (n = {int(t[t.category == c].n.iloc[0])})" for c in cats])
ax.tick_params(axis='both', which='both', length=0)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font); label.set_fontsize(FONT_SIZE_TICKS)
for spine in ax.spines.values():
    spine.set_visible(False)
cbar = fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax, fraction=0.05, pad=0.04)
cbar.set_label("Median Δ Spearman\n(fine-tuned − base)", fontsize=FONT_SIZE_TICKS, fontproperties=arial_font)
cbar.ax.tick_params(labelsize=FONT_SIZE_TICKS - 2, width=SPINE_WIDTH)
for label in cbar.ax.get_yticklabels():
    label.set_fontproperties(arial_font); label.set_fontsize(FONT_SIZE_TICKS - 2)
cbar.outline.set_linewidth(SPINE_WIDTH)
plt.tight_layout()
plt.savefig("S5B_gain_by_taxon_2026-10.svg", dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
#@title S5C: Spearman gain from fine-tuning by DMS selection type, all model sizes (ProteinGym test)
# Cells: median over assays of (fine-tuned - base) Spearman; stars: two-sided Wilcoxon signed-rank test of fine-tuned vs base
# within the category (* p < 0.05, ** p < 0.01, *** p < 0.001). Color scale shared by S5B (taxon) and S5C (selection type).
# Extends Fig 5C/D (650M) to all model sizes.
PANEL_GROUP = 'coarse_selection_type'
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Rectangle
from scipy.stats import wilcoxon

DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 15
FONT_SIZE_CELL = 12
SPINE_WIDTH = 1.2
SIZES = ['8M', '35M', '150M', '650M']
GROUPS = {'taxon': ['Prokaryote', 'Eukaryote', 'Human', 'Virus'],
          'coarse_selection_type': ['Stability', 'Activity', 'OrganismalFitness', 'Expression', 'Binding']}
NICE = {'OrganismalFitness': 'Organismal fitness'}
arial_font = fm.FontProperties(fname=ARIAL)

pg = pd.read_csv(f"proteingym_scores_esm2_husc.csv")
pg = pg[pg.split == 'test'].reset_index(drop=True)

def stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''

rows = []
for col, cats in GROUPS.items():
    for c in cats:
        sub = pg[pg[col] == c]
        for s in SIZES:
            d = sub[f'ESM_{s}_ft_spearman'] - sub[f'ESM_{s}_base_spearman']
            rows.append(dict(group=col, category=c, n=len(sub), size=s, median_delta=d.median(),
                             p=wilcoxon(sub[f'ESM_{s}_ft_spearman'], sub[f'ESM_{s}_base_spearman']).pvalue))
tab = pd.DataFrame(rows)
vmax = np.abs(tab.median_delta).max()                        # shared by S5B and S5C
norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
cmap = plt.get_cmap('coolwarm')
cats = GROUPS[PANEL_GROUP]
t = tab[tab.group == PANEL_GROUP]
print(t.pivot_table(index=['category', 'n'], columns='size', values='median_delta', sort=False)[SIZES].round(3).to_string())
print(t.pivot_table(index='category', columns='size', values='p', sort=False)[SIZES].to_string(float_format=lambda v: f"{v:.1e}"))

fig, ax = plt.subplots(figsize=(7.5, 1.1 * len(cats) + 1.4), dpi=DPI)
for i, c in enumerate(cats):
    for j, s in enumerate(SIZES):
        r = t[(t.category == c) & (t['size'] == s)].iloc[0]
        v = r.median_delta
        ax.add_patch(Rectangle((j - 0.5, i - 0.5), 1, 1, facecolor=cmap(norm(v)), edgecolor='white', linewidth=1.5))
        txt = "0.00" if round(v, 2) == 0 else f"{v:+.2f}".replace('-', '−')
        ax.text(j, i, txt + stars(r.p), ha='center', va='center', fontproperties=arial_font, fontsize=FONT_SIZE_CELL,
                color='white' if abs(norm(v) - 0.5) > 0.35 else 'black')
ax.set_xlim(-0.5, len(SIZES) - 0.5)
ax.set_ylim(len(cats) - 0.5, -0.5)
ax.set_aspect('equal')
ax.set_xticks(range(len(SIZES))); ax.set_xticklabels(SIZES)
ax.xaxis.tick_top()
ax.set_xlabel("Model Size", fontsize=FONT_SIZE_LABELS, fontproperties=arial_font, labelpad=10)
ax.xaxis.set_label_position('top')
ax.set_yticks(range(len(cats)))
ax.set_yticklabels([f"{NICE.get(c, c)} (n = {int(t[t.category == c].n.iloc[0])})" for c in cats])
ax.tick_params(axis='both', which='both', length=0)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font); label.set_fontsize(FONT_SIZE_TICKS)
for spine in ax.spines.values():
    spine.set_visible(False)
cbar = fig.colorbar(plt.cm.ScalarMappable(cmap=cmap, norm=norm), ax=ax, fraction=0.05, pad=0.04)
cbar.set_label("Median Δ Spearman\n(fine-tuned − base)", fontsize=FONT_SIZE_TICKS, fontproperties=arial_font)
cbar.ax.tick_params(labelsize=FONT_SIZE_TICKS - 2, width=SPINE_WIDTH)
for label in cbar.ax.get_yticklabels():
    label.set_fontproperties(arial_font); label.set_fontsize(FONT_SIZE_TICKS - 2)
cbar.outline.set_linewidth(SPINE_WIDTH)
plt.tight_layout()
plt.savefig("S5C_gain_by_selection_type_2026-10.svg", dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
#@title S6A: wild-type NLL before and after HuSC fine-tuning, all model sizes (ProteinGym test)
# S6A body: mean wild-type NLL per ProteinGym TEST assay (unmasked forward pass), base vs HuSC fine-tuned ESM2, all sizes
# (wt_nll_esm2_{size}_husc.csv). Extends Fig 6A (650M).
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy.stats import wilcoxon

FIGURE_SIZE = (8, 6)
DPI = 300
FONT_SIZE_LABELS = 18
FONT_SIZE_TICKS = 18
FONT_SIZE_TEXT = 14
FONT_SIZE_LEGEND = 14
SPINE_WIDTH = 1.2
SIZES = ['8M', '35M', '150M', '650M']
MODEL_COLORS = {"Base": '#708090', "Fine-Tuned": '#d62728'}     # as Fig 5B / 6A-C
arial_font = fm.FontProperties(fname=ARIAL)

rows = []
for s in SIZES:
    n = pd.read_csv(f"wt_nll_esm2_{s}_husc.csv")
    m = n[n.split == "test"].groupby("DMS_id")[["nll_base", "nll_ft"]].mean().reset_index()
    rows.append(m.assign(size=s))
wide = pd.concat(rows)
wide.to_csv("S6A_wt_nll_by_size_2026-10.csv", index=False)
long = pd.concat([wide.assign(model=m, nll=wide[c]) for m, c in [("Base", "nll_base"), ("Fine-Tuned", "nll_ft")]])

tests = {}
for s in SIZES:
    w = wide[wide["size"] == s]
    tests[s] = wilcoxon(w.nll_ft, w.nll_base)
    print(f"{s:5s} n = {len(w)}  median base {w.nll_base.median():.3f} -> fine-tuned {w.nll_ft.median():.3f}  "
          f"increased in {(w.nll_ft > w.nll_base).sum()}/{len(w)}  p = {tests[s].pvalue:.1e}")

fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)
sns.violinplot(x="size", y="nll", hue="model", data=long, order=SIZES, hue_order=list(MODEL_COLORS), palette=MODEL_COLORS,
               cut=0, inner="box", linewidth=SPINE_WIDTH + 0.5, ax=ax)
ymax = long.nll.max(); yoffset = 0.05 * ymax
for i, s in enumerate(SIZES):
    p = tests[s].pvalue
    p_text = "p < 0.001" if p < 0.001 else f"p = {p:.3f}"
    ax.plot([i - 0.2, i - 0.2, i + 0.2, i + 0.2], [ymax + yoffset, ymax + 1.3 * yoffset, ymax + 1.3 * yoffset, ymax + yoffset],
            color="black", linewidth=1)
    ax.text(i, ymax + 1.5 * yoffset, p_text, ha="center", va="bottom", fontsize=FONT_SIZE_TEXT, fontproperties=arial_font)
ax.set_ylim(0, ymax + 4 * yoffset)
ax.set_xlabel("Model Size", fontsize=FONT_SIZE_LABELS, fontweight="bold", fontproperties=arial_font)
ax.set_ylabel("Mean wild-type NLL per assay", fontsize=FONT_SIZE_LABELS, fontweight="bold", fontproperties=arial_font,
              labelpad=10)
ax.legend(title=None, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=2,
          prop=fm.FontProperties(fname=ARIAL, size=FONT_SIZE_LEGEND))
ax.tick_params(axis="both", which="major", width=SPINE_WIDTH, color="black", bottom=True, left=True)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font); label.set_fontsize(FONT_SIZE_TICKS)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["left", "bottom"]:
    ax.spines[spine].set_linewidth(SPINE_WIDTH); ax.spines[spine].set_color("black")
plt.tight_layout()
plt.savefig("S6A_wt_nll_by_size_2026-10.svg", dpi=DPI, bbox_inches='tight')
plt.show()

In [ ]:
#@title S6B: change in wild-type NLL by site-tolerance quintile, HuSC fine-tuned (650M, unmasked)
# Change in wild-type NLL (fine-tuned - base ESM2-650M, unmasked forward pass) by site-tolerance quintile (TEST assays).
# Per assay, sites (>= 3 single substitutions) split into tolerance quintiles (Q1 least tolerant);
# value = mean ΔNLL over the quintile's sites; points = median over assays, bars = IQR.
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from scipy.stats import spearmanr
FIGURE_SIZE = (7, 6); DPI = 300; FONT_SIZE_LABELS = 18; FONT_SIZE_TICKS = 16
SPINE_WIDTH = 1.2
arial_font = fm.FontProperties(fname=ARIAL)
MIN_SUBS, MIN_SINGLES, MIN_SITES, NQ = 3, 20, 10, 5
COLOR = '#d62728'


def site_tolerance(dms_df):
    singles = dms_df[~dms_df["mutant"].str.contains(":")]
    if len(singles) < MIN_SINGLES:
        return None
    pos = singles["mutant"].str.extract(r"[A-Z](\d+)[A-Z]")[0].astype(int) - 1
    site = singles.groupby(pos)["DMS_score"].agg(["mean", "size"])
    return site.loc[site["size"] >= MIN_SUBS, "mean"]


test_ids = set(pd.read_csv(f"proteingym_dev_test_split.csv").query("split == 'test'").DMS_id)
rows = []
nll_df = pd.read_csv(f"wt_nll_esm2_650M_husc.csv")
nll_df = nll_df[nll_df.DMS_id.isin(test_ids)]
for dms_id, nll in nll_df.groupby("DMS_id"):
    tol = site_tolerance(pd.read_csv(f"DMS/{dms_id}.csv", usecols=["mutant", "DMS_score"]))
    if tol is None:
        continue
    sites = nll.set_index("position").join(tol.rename("tolerance"), how="inner")
    if len(sites) < MIN_SITES:
        continue
    qn = np.minimum(np.ceil(sites["tolerance"].rank(pct=True) * NQ).astype(int), NQ)
    for qi, g in sites.groupby(qn):
        rows.append(dict(DMS_id=dms_id, quintile=int(qi), delta_nll=(g.nll_ft - g.nll_base).mean()))
tab = pd.DataFrame(rows)
tab.to_csv("S6B_delta_nll_by_tolerance_quintile_650M_2026-10.csv", index=False)
summ = tab.groupby("quintile").delta_nll.quantile([0.25, 0.5, 0.75]).unstack()
print(f"n assays = {tab.DMS_id.nunique()}")
print(summ.round(3).to_string())
r = spearmanr(tab.quintile, tab.delta_nll)
print(f"Spearman(quintile, ΔNLL) = {r.statistic:+.2f}, p = {r.pvalue:.1e}")

fig, ax = plt.subplots(figsize=FIGURE_SIZE, dpi=DPI)
x = np.arange(1, NQ + 1)
ax.errorbar(x, summ[0.5], yerr=[summ[0.5] - summ[0.25], summ[0.75] - summ[0.5]], fmt="o-", color=COLOR, ecolor=COLOR,
            markersize=9, linewidth=2.2, capsize=4, markeredgecolor="black", markeredgewidth=0.6)
ax.axhline(0, color="black", linestyle="--", linewidth=1, zorder=0)
ax.set_xticks(range(1, NQ + 1)); ax.set_xticklabels([f"Q{i}" for i in range(1, NQ + 1)])
ax.set_xlabel("Site-tolerance quintile\n(Q1 least → Q5 most tolerant)", fontsize=FONT_SIZE_LABELS, fontweight="bold",
              fontproperties=arial_font)
ax.set_ylabel("Δ NLL (fine-tuned − base)", fontsize=FONT_SIZE_LABELS, fontweight="bold", fontproperties=arial_font,
              labelpad=10)
ax.tick_params(axis="both", which="major", width=SPINE_WIDTH, color="black", bottom=True, left=True)
for label in ax.get_xticklabels() + ax.get_yticklabels():
    label.set_fontproperties(arial_font); label.set_fontsize(FONT_SIZE_TICKS)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
for spine in ["left", "bottom"]:
    ax.spines[spine].set_linewidth(SPINE_WIDTH); ax.spines[spine].set_color("black")
plt.tight_layout()
plt.savefig("S6B_delta_nll_by_tolerance_quintile_650M_2026-10.svg", dpi=DPI, bbox_inches="tight")
plt.show()

In [ ]:
#@title S6C setup: masked-marginal wild-type NLL (base and HuSC fine-tuned ESM2-650M), TEST assays
# For each ProteinGym TEST target, mask one position at a time and take -log p(wild-type residue) at the masked position
# (full-vocabulary softmax, same as the unmasked wt_nll_esm2_650M_husc.csv). Only positions used in S6 are scored:
# sites with >= 3 single substitutions in at least one of the target's TEST assays. Base = LoRA adapter disabled.
# Output: wt_nll_masked_esm2_650M_husc.csv (DMS_id, position, nll_base, nll_ft). If that file exists the cell only
# reports it: the forward passes are not rerun.
# To recompute, delete the file (the per-target partial file makes a recompute resumable). Needs a GPU to recompute.
import os
import pandas as pd

OUT, PART = "wt_nll_masked_esm2_650M_husc.csv", "wt_nll_masked_esm2_650M_husc_partial.csv"

if not os.path.exists(OUT):
    !pip uninstall -y -q torchao
    import gc
    import numpy as np, torch
    from transformers import EsmForMaskedLM, EsmTokenizer
    from peft import LoraConfig, get_peft_model
    from tqdm.auto import tqdm

    assert torch.cuda.is_available(), "switch the runtime to a GPU"
    device = torch.device("cuda")
    MODEL = "facebook/esm2_t33_650M_UR50D"
    LORA = LoraConfig(task_type="FEATURE_EXTRACTION", r=8, lora_alpha=64, lora_dropout=0.1,
                      target_modules=["query", "key", "value", "output.dense"])
    CKPT = "finetuned_esm2_650M_husc.pth"
    BATCH, MIN_SUBS = 32, 3

    tokenizer = EsmTokenizer.from_pretrained(MODEL)
    summ = pd.read_csv("DMS_substitutions.csv").drop_duplicates("DMS_id").set_index("DMS_id")
    split = pd.read_csv("proteingym_dev_test_split.csv")
    test_ids = split[split.split == "test"].DMS_id.tolist()

    # positions to score per target sequence (union over that target's TEST assays)
    pos_by_seq, assays_by_seq = {}, {}
    for d in test_ids:
        seq = summ.at[d, "target_seq"]
        dms = pd.read_csv(f"DMS/{d}.csv", usecols=["mutant", "DMS_score"])
        singles = dms[~dms.mutant.str.contains(":")]
        pos = singles.mutant.str.extract(r"[A-Z](\d+)[A-Z]")[0].astype(int) - 1
        pos_by_seq.setdefault(seq, set()).update(pos.value_counts().loc[lambda s: s >= MIN_SUBS].index)
        assays_by_seq.setdefault(seq, []).append(d)
    seqs = sorted(pos_by_seq)
    print(f"{len(seqs)} target sequences, {sum(len(v) for v in pos_by_seq.values()):,} positions to score per model")

    done = pd.read_csv(PART) if os.path.exists(PART) else pd.DataFrame(columns=["seq_idx", "position", "arm", "nll"])
    done_keys = set(zip(done.seq_idx, done.arm))

    @torch.no_grad()
    def masked_wt_nll(model, seq, positions):
        ids = tokenizer(seq, return_tensors="pt")["input_ids"][0]
        out = []
        for i in range(0, len(positions), BATCH):
            chunk = positions[i:i + BATCH]
            batch = ids.repeat(len(chunk), 1)
            for k, p in enumerate(chunk):
                batch[k, p + 1] = tokenizer.mask_token_id          # +1 for <cls>
            lp = torch.log_softmax(model(batch.to(device)).logits.float(), dim=-1)
            for k, p in enumerate(chunk):
                out.append(-lp[k, p + 1, ids[p + 1]].item())
        return out

    model = get_peft_model(EsmForMaskedLM.from_pretrained(MODEL), LORA).to(device).eval()
    missing, unexpected = model.load_state_dict(torch.load(CKPT, map_location=device), strict=False)
    assert not unexpected and not [k for k in missing if "lora" in k], CKPT
    for si, seq in enumerate(tqdm(seqs)):
        positions = sorted(p for p in pos_by_seq[seq] if p < len(seq))
        rows = []
        for a in ["ft", "base"]:
            if (si, a) in done_keys:
                continue
            if a == "base":
                with model.disable_adapter():
                    vals = masked_wt_nll(model, seq, positions)
            else:
                vals = masked_wt_nll(model, seq, positions)
            rows += [dict(seq_idx=si, position=p, arm=a, nll=v) for p, v in zip(positions, vals)]
        if rows:
            pd.DataFrame(rows).to_csv(PART, mode="a", header=not os.path.exists(PART), index=False)
    del model; gc.collect(); torch.cuda.empty_cache()

    part = pd.read_csv(PART).drop_duplicates(["seq_idx", "position", "arm"])
    wide = part.pivot_table(index=["seq_idx", "position"], columns="arm", values="nll").reset_index()
    wide = wide.rename(columns={a: f"nll_{a}" for a in ["base", "ft"]})
    out = pd.concat([wide[wide.seq_idx == si].assign(DMS_id=d) for si, seq in enumerate(seqs) for d in assays_by_seq[seq]])
    out[["DMS_id", "position", "nll_base", "nll_ft"]].to_csv(OUT, index=False)

out = pd.read_csv(OUT)
print(f"{OUT}: {len(out):,} rows, {out.DMS_id.nunique()} assays (saved; forward passes not rerun)")
m = out.groupby("DMS_id")[["nll_base", "nll_ft"]].mean()
print("median per-assay masked WT NLL  base %.3f  HuSC fine-tuned %.3f" % tuple(m.median()))

In [ ]:
#@title S6C: change in wild-type NLL at intolerant vs tolerant sites, HuSC fine-tuned, unmasked vs masked-marginal scoring (650M)
# Change in wild-type NLL (fine-tuned - base ESM2-650M) at variation-intolerant / -tolerant sites, scored from an unmasked
# forward pass (left; wt_nll_esm2_650M_husc.csv, as in Fig 6B) and by masked marginals (right;
# wt_nll_masked_esm2_650M_husc.csv from "S6C setup", each site masked in turn). P-values: tolerant vs intolerant
# (two-sided Wilcoxon signed-rank, paired by assay).
import os
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from scipy.stats import wilcoxon
FIGURE_SIZE = (12, 6); DPI = 300; FONT_SIZE_LABELS = 18; FONT_SIZE_TICKS = 16; FONT_SIZE_TEXT = 15
SPINE_WIDTH = 1.2
arial_font = fm.FontProperties(fname=ARIAL)
MIN_SUBS, MIN_SINGLES, MIN_SITES, TAIL = 3, 20, 10, 0.2
GROUPS = ["Variation Intolerant Sites", "Variation Tolerant Sites"]
GROUP_LABELS = ["Variation\nIntolerant Sites", "Variation\nTolerant Sites"]
COLOR = '#d62728'
SCORING = {"Unmasked": "wt_nll_esm2_650M_husc.csv", "Masked marginal": "wt_nll_masked_esm2_650M_husc.csv"}


def site_tolerance(dms_df):
    singles = dms_df[~dms_df["mutant"].str.contains(":")]
    if len(singles) < MIN_SINGLES:
        return None
    pos = singles["mutant"].str.extract(r"[A-Z](\d+)[A-Z]")[0].astype(int) - 1   # 1-indexed -> 0-indexed
    site = singles.groupby(pos)["DMS_score"].agg(["mean", "size"])
    return site.loc[site["size"] >= MIN_SUBS, "mean"]


test_ids = set(pd.read_csv(f"proteingym_dev_test_split.csv").query("split == 'test'").DMS_id)
tol_cache = {}
records = []
for scoring, fname in SCORING.items():
    if not os.path.exists(f"{fname}"):
        print(f"{fname} not found: {scoring} panel left empty")
        continue
    nll_df = pd.read_csv(f"{fname}")
    nll_df = nll_df[nll_df.DMS_id.isin(test_ids)]
    for dms_id, nll in nll_df.groupby("DMS_id"):
        if dms_id not in tol_cache:
            tol_cache[dms_id] = site_tolerance(pd.read_csv(f"DMS/{dms_id}.csv", usecols=["mutant", "DMS_score"]))
        tol = tol_cache[dms_id]
        if tol is None:
            continue
        sites = nll.set_index("position").join(tol.rename("tolerance"), how="inner")
        if len(sites) < MIN_SITES:
            continue
        q = sites["tolerance"].rank(pct=True)
        for label, mask in [(GROUPS[0], q <= TAIL), (GROUPS[1], q > 1 - TAIL)]:
            sel = sites[mask]
            records.append(dict(scoring=scoring, DMS_id=dms_id, group=label, delta_nll=(sel.nll_ft - sel.nll_base).mean()))
df = pd.DataFrame(records).dropna()
df.to_csv("S6C_delta_nll_tolerance_unmasked_vs_masked_650M_2026-10.csv", index=False)

tests = {}
for scoring in df.scoring.unique():
    w = df[df.scoring == scoring].pivot(index="DMS_id", columns="group", values="delta_nll").dropna()
    diff = w[GROUPS[1]] - w[GROUPS[0]]
    tests[scoring] = wilcoxon(diff)
    print(f"{scoring:16s} n = {len(w)}  median ΔNLL intolerant {w[GROUPS[0]].median():+.3f}, "
          f"tolerant {w[GROUPS[1]].median():+.3f}; tolerant > intolerant in {(diff > 0).sum()}/{len(diff)}; "
          f"p = {tests[scoring].pvalue:.1e}")

fig, axes = plt.subplots(1, 2, figsize=FIGURE_SIZE, dpi=DPI, sharey=True)
ymax, ymin = df.delta_nll.max(), df.delta_nll.min(); yoffset = 0.05 * (ymax - ymin)
for ax, scoring in zip(axes, SCORING):
    sub = df[df.scoring == scoring]
    if len(sub):
        sns.violinplot(x="group", y="delta_nll", data=sub, order=GROUPS, color=COLOR, cut=0, inner="box",
                       linewidth=SPINE_WIDTH + 0.5, width=0.6, ax=ax)
        p = tests[scoring].pvalue
        p_text = "p < 0.001" if p < 0.001 else f"p = {p:.3f}"
        ax.plot([0, 0, 1, 1], [ymax + yoffset, ymax + 1.3 * yoffset, ymax + 1.3 * yoffset, ymax + yoffset],
                color="black", linewidth=1)
        ax.text(0.5, ymax + 1.5 * yoffset, p_text, ha="center", va="bottom", fontsize=FONT_SIZE_TEXT,
                fontproperties=arial_font)
    ax.axhline(0, color="black", linestyle="--", linewidth=1, zorder=0)
    ax.text(0.5, 1.02, scoring, transform=ax.transAxes, ha="center", va="bottom", fontsize=FONT_SIZE_LABELS,
            fontweight="bold", fontproperties=arial_font)
    ax.set_xlim(-0.5, 1.5); ax.set_xticks(range(2)); ax.set_xticklabels(GROUP_LABELS)
    ax.set_xlabel("")
    ax.tick_params(axis="both", which="major", width=SPINE_WIDTH, color="black", bottom=True, left=True)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontproperties(arial_font); label.set_fontsize(FONT_SIZE_TICKS)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)
    for spine in ["left", "bottom"]:
        ax.spines[spine].set_linewidth(SPINE_WIDTH); ax.spines[spine].set_color("black")
axes[0].set_ylabel("Δ NLL (fine-tuned − base)", fontsize=FONT_SIZE_LABELS, fontweight="bold", fontproperties=arial_font,
                   labelpad=10)
axes[1].set_ylabel("")
axes[0].set_ylim(top=ymax + 4 * yoffset)
plt.tight_layout()
plt.savefig("S6C_delta_nll_tolerance_unmasked_vs_masked_650M_2026-10.svg", dpi=DPI, bbox_inches="tight")
plt.show()